# LinearRAG Evaluation — 1887 French Parliamentary Corpus

**Method**: LinearRAG (ICLR 2026 SOTA graph-based RAG)  
**Embeddings**: Cohere `embed-v4.0` (via `CohereEmbeddingModel`)  
**LLM**: Cohere `command-r-plus` (via `LLM_Model`)  
**NER**: `fr_core_news_lg` (French SpaCy)

## Workflow
1. Load the three 1887 corpora (Les Débats, L'Intransigeant, Le Gaulois)
2. Build the LinearRAG tri-graph index (offline — resumable via parquet cache)
3. Load the 5 multi-hop question datasets
4. Run LinearRAG retrieval for all questions
5. Evaluate Recall@K, Accuracy@K and MRR@K
6. Break down metrics by question type and source pair

In [1]:
# ============================================================
# Cell 1 — Setup & imports
# ============================================================
import sys, os

# ── Project root: robust detection regardless of kernel cwd ──
# VS Code Jupyter kernels may start with cwd = project root OR notebooks/
_cwd = os.getcwd()
if os.path.exists(os.path.join(_cwd, "scripts")):
    PROJECT_ROOT = _cwd                                    # cwd is already the project root
elif os.path.exists(os.path.join(_cwd, "..", "scripts")):
    PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))  # cwd is notebooks/
else:
    raise RuntimeError(f"Cannot locate project root from cwd={_cwd!r}")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")

sys.path.insert(0, PROJECT_ROOT)

# LinearRAG source directory
LINEARRAG_SRC = os.path.join(PROJECT_ROOT, "scripts", "SOTA", "LinearRAG-main")
sys.path.insert(0, LINEARRAG_SRC)

# Clear any cached import failures for LinearRAG modules so re-running
# this cell (without a kernel restart) always picks up the correct paths.
for _mod in list(sys.modules.keys()):
    if _mod == "src" or _mod.startswith("src."):
        del sys.modules[_mod]

# Load .env
from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join(PROJECT_ROOT, ".env"))

# Project config
from scripts.config import DATA_DIR, COHERE_API_KEY, COHERE_EMBEDDING_MODEL

# LinearRAG components (adapted for Cohere + French)
from src.api_embedding import CohereEmbeddingModel
from src.utils import LLM_Model
from src.config import LinearRAGConfig
from src.LinearRAG import LinearRAG

# Question / evaluation utilities
from scripts.utils.questions import create_questions_dataset
from scripts.retrieval_utils import (
    find_gold_ranks,
    get_existing_ids,
    evaluate_and_print_metrics,
    compute_recall_metrics_dataframe,
    compute_accuracy_at_k,
    compute_mrr_at_k,
)

import json
import regex as re
import pandas as pd
import numpy as np
from tqdm import tqdm

print(f"LINEARRAG_SRC : {LINEARRAG_SRC}")
print("All imports successful")


PROJECT_ROOT : c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026


c:\Users\plathoon\AppData\Local\anaconda3\envs\debattre\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LINEARRAG_SRC : c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\scripts\SOTA\LinearRAG-main
All imports successful


In [2]:
# ============================================================
# Cell 2 — Configuration
# ============================================================

# ─── Paths ────────────────────────────────────────────────────
QUESTIONS_DIR   = os.path.join(DATA_DIR, "questions")
INDEX_DIR       = os.path.join(DATA_DIR, "linearrag_index")
OUTPUT_DIR      = os.path.join(DATA_DIR, "retrieval")
OUTPUT_FILE     = os.path.join(OUTPUT_DIR, "linearrag_retrieval_results.jsonl")

# Corpus source directories (same as build_indexes.py)
DEBATS_SPLIT_DIR   = os.path.join(DATA_DIR, "corpus_1887_splitted_v2")
LINTRANSIGEANT_DIR = os.path.join(DATA_DIR, "lintransigeant_1887")
LEGAULOIS_DIR      = os.path.join(DATA_DIR, "legaulois_1887")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INDEX_DIR,  exist_ok=True)

# ─── Model settings ────────────────────────────────────────────
EMBEDDING_MODEL_NAME = COHERE_EMBEDDING_MODEL   # "embed-v4.0"
LLM_MODEL_NAME       = "command-r-plus"
SPACY_MODEL          = "fr_core_news_lg"

# ─── LinearRAG hyperparams ─────────────────────────────────────
DATASET_NAME        = "1887_combined"
RETRIEVAL_TOP_K     = 10
MAX_ITERATIONS      = 3
TOP_K_SENTENCE      = 3
ITERATION_THRESHOLD = 0.4
PASSAGE_RATIO       = 2.0

# ─── Evaluation settings ───────────────────────────────────────
K_VALUES = [1, 2, 3, 5, 10]

# ─── Question datasets (same mapping as notebook 75) ───────────
QUESTION_DATASETS = [
    {
        "name": "Generic_LeGaulois_Debats",
        "file": "MH_Generic_legaulois_v1.jsonl",
        "column_mapping": {
            "id_1": "journal_id", "id_2": "debat_id",
            "doc_1": "journal_doc", "doc_2": "debat_doc",
            "metadata_1": "journal_metadata", "metadata_2": "debat_metadata",
        },
        "source1_name": "Le Gaulois",
        "source2_name": "Les Débats",
        "question_type": "MH - Generic",
    },
    {
        "name": "Generic_LIntransigeant_Debats",
        "file": "generated_questions_lintransigeant_all.jsonl",
        "column_mapping": {
            "id_1": "journal_id", "id_2": "debat_id",
            "doc_1": "journal_doc", "doc_2": "debat_doc",
            "metadata_1": "journal_metadata", "metadata_2": "debat_metadata",
        },
        "source1_name": "L'Intransigeant",
        "source2_name": "Les Débats",
        "question_type": "MH - Generic",
    },
    {
        "name": "BridgeEntity_LIntransigeant_Debats",
        "file": "MH_BridgeEntity_lintransigeant_v2.jsonl",
        "column_mapping": {
            "id_1": "journal_id", "id_2": "debat_id",
            "doc_1": "journal_doc", "doc_2": "debat_doc",
            "metadata_1": "journal_metadata", "metadata_2": "debat_metadata",
        },
        "source1_name": "L'Intransigeant",
        "source2_name": "Les Débats",
        "question_type": "MH - BridgeEntity",
    },
    {
        "name": "Comparative_LeGaulois_LIntransigeant",
        "file": "MH_Comparative_legaulois_lintransigeant_v3.jsonl",
        "column_mapping": {
            "id_1": "legaulois_id", "id_2": "lintransigeant_id",
            "doc_1": "legaulois_doc", "doc_2": "lintransigeant_doc",
            "metadata_1": "legaulois_metadata", "metadata_2": "lintransigeant_metadata",
        },
        "source1_name": "Le Gaulois",
        "source2_name": "L'Intransigeant",
        "question_type": "MH - Comparative",
    },
    {
        "name": "Generic_LeGaulois_LIntransigeant",
        "file": "MH_Generic_legaulois_lintransigeant_v3.jsonl",
        "column_mapping": {
            "id_1": "legaulois_id", "id_2": "lintransigeant_id",
            "doc_1": "legaulois_doc", "doc_2": "lintransigeant_doc",
            "metadata_1": "legaulois_metadata", "metadata_2": "lintransigeant_metadata",
        },
        "source1_name": "Le Gaulois",
        "source2_name": "L'Intransigeant",
        "question_type": "MH - Generic",
    },
]

print(f"Index dir    : {INDEX_DIR}")
print(f"Output file  : {OUTPUT_FILE}")
print(f"Datasets     : {len(QUESTION_DATASETS)}")
print(f"Top-K        : {RETRIEVAL_TOP_K}  |  K_VALUES: {K_VALUES}")

Index dir    : c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\linearrag_index
Output file  : c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\retrieval\linearrag_retrieval_results.jsonl
Datasets     : 5
Top-K        : 10  |  K_VALUES: [1, 2, 3, 5, 10]


In [3]:
# ============================================================
# Cell 3 — Corpus loading helpers
# Mirrors scripts/build_indexes.py exactly.
# ============================================================

_LINTRANSIGEANT_SECTION_RE = re.compile(r"La Chambre\n.*")
_LEGAULOIS_SECTION_RE      = re.compile(
    r"CHAMBRE DES D[ÉE]PUT[ÉE]S.{50,}?[A-ZÉ\s\n]{4,}", re.DOTALL
)


def collect_debats_chunks(split_dir: str) -> list:
    """Return [(filename, text)] for all pre-chunked Les Débats files.
    filename is kept with .txt extension, e.g. '1887-01-11_003_000.txt'.
    """
    result = []
    for folder in sorted(os.listdir(split_dir)):
        folder_path = os.path.join(split_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            fpath = os.path.join(folder_path, fname)
            with open(fpath, encoding="utf-8") as fh:
                result.append((fname, fh.read()))
    return result


def load_newspaper_corpus(corpus_dir: str) -> list:
    """Return [(base_name, text)] for all newspaper article files.
    base_name = filename without extension, e.g. '18870113'.
    Applies escape-sequence cleanup as in build_indexes.py.
    """
    result = []
    for folder in sorted(os.listdir(corpus_dir)):
        folder_path = os.path.join(corpus_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        files = os.listdir(folder_path)
        filepath = os.path.join(folder_path, files[0])
        with open(filepath, encoding="utf-8") as fh:
            text = fh.read().replace("\\n", "\n").replace("\\t", "\t").replace("\\r", "\r")
        result.append((files[0].split(".")[0], text))
    return result


def extract_la_chambre(corpus: list) -> list:
    result = []
    for name, text in corpus:
        matches = _LINTRANSIGEANT_SECTION_RE.findall(text)
        if matches:
            result.append((name, matches[0]))
    print(f"  'La Chambre' extracted : {len(result)} / {len(corpus)}")
    return result


def extract_chambre_deputes(corpus: list) -> list:
    result = []
    for name, text in corpus:
        matches = _LEGAULOIS_SECTION_RE.findall(text)
        if matches:
            result.append((name, matches[0]))
    print(f"  'Chambre des Députés' extracted : {len(result)} / {len(corpus)}")
    return result


print("Corpus loading helpers defined")

Corpus loading helpers defined


In [4]:
# ============================================================
# Cell 4 — Load all three corpora
# ============================================================
print("Loading corpora...\n")

# Les Débats parlementaires (pre-chunked by chunk_debats.py)
print("[1/3] Les Débats (pre-chunked)...")
debats = collect_debats_chunks(DEBATS_SPLIT_DIR)
print(f"  {len(debats)} chunks loaded")

# L'Intransigeant — extract 'La Chambre' section
print("\n[2/3] L'Intransigeant...")
lintransigeant_raw = load_newspaper_corpus(LINTRANSIGEANT_DIR)
print(f"  {len(lintransigeant_raw)} raw files loaded")
lintransigeant = extract_la_chambre(lintransigeant_raw)

# Le Gaulois — extract 'CHAMBRE DES DÉPUTÉS' section
print("\n[3/3] Le Gaulois...")
legaulois_raw = load_newspaper_corpus(LEGAULOIS_DIR)
print(f"  {len(legaulois_raw)} raw files loaded")
legaulois = extract_chambre_deputes(legaulois_raw)

# ── Combine into a single list of (orig_id, text, source) ──
all_docs = (
    [(orig_id, text, "debats")        for orig_id, text in debats] +
    [(orig_id, text, "lintransigeant") for orig_id, text in lintransigeant] +
    [(orig_id, text, "legaulois")      for orig_id, text in legaulois]
)

print(f"\nTotal passages : {len(all_docs)}")
print(f"  Débats: {len(debats)}  |  Intransigeant: {len(lintransigeant)}  |  Gaulois: {len(legaulois)}")

# ── Build index-prefixed texts for adjacent-passage edge construction ──
# LinearRAG.add_adjacent_passage_edges() uses r'^(\d+):' to determine ordering.
# We prepend "<index>: " so that temporally adjacent chunks are linked in the graph.
indexed_texts = [f"{i}: {text}" for i, (_, text, _) in enumerate(all_docs)]

# Mapping: indexed text → original document ID (used after retrieval to recover IDs)
text_to_orig_id = {indexed_texts[i]: orig_id for i, (orig_id, _, _) in enumerate(all_docs)}

print(f"\nIndexed passage texts built: {len(indexed_texts)}")

Loading corpora...

[1/3] Les Débats (pre-chunked)...
  5103 chunks loaded

[2/3] L'Intransigeant...
  365 raw files loaded
  'La Chambre' extracted : 79 / 365

[3/3] Le Gaulois...
  365 raw files loaded
  'Chambre des Députés' extracted : 78 / 365

Total passages : 5260
  Débats: 5103  |  Intransigeant: 79  |  Gaulois: 78

Indexed passage texts built: 5260


In [5]:
# ============================================================
# Cell 5 — Initialise LinearRAG models and config
# ============================================================
print("Initialising LinearRAG components...")

embedding_model = CohereEmbeddingModel(
    api_key=COHERE_API_KEY,
    model=EMBEDDING_MODEL_NAME,
    batch_size=96,
)

llm_model = LLM_Model(
    llm_model=LLM_MODEL_NAME,
    api_key=COHERE_API_KEY,
)

config = LinearRAGConfig(
    dataset_name=DATASET_NAME,
    embedding_model=embedding_model,
    llm_model=llm_model,
    spacy_model=SPACY_MODEL,
    working_dir=INDEX_DIR,
    retrieval_top_k=RETRIEVAL_TOP_K,
    batch_size=96,
    max_iterations=MAX_ITERATIONS,
    top_k_sentence=TOP_K_SENTENCE,
    iteration_threshold=ITERATION_THRESHOLD,
    passage_ratio=PASSAGE_RATIO,
)

linearrag = LinearRAG(config)
print("LinearRAG ready")

Initialising LinearRAG components...
[passage] Loaded 5260 records from c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\linearrag_index\1887_combined\passage_embedding.parquet
[entity] Loaded 95114 records from c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\linearrag_index\1887_combined\entity_embedding.parquet
[sentence] Loaded 119623 records from c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\linearrag_index\1887_combined\sentence_embedding.parquet
LinearRAG ready


In [6]:
# ============================================================
# Cell 6 — Offline indexing (resumable)
# Parquet caches embeddings; NER results cached as JSON.
# Re-running this cell is safe — only NEW passages are re-embedded.
# ============================================================
import pandas as _pd

passage_parquet = os.path.join(INDEX_DIR, DATASET_NAME, "passage_embedding.parquet")
ner_json        = os.path.join(INDEX_DIR, DATASET_NAME, "ner_results.json")

# ── Show current index status before doing any work ──────────
n_total = len(indexed_texts)
n_passage_cached = len(_pd.read_parquet(passage_parquet)) if os.path.exists(passage_parquet) else 0
n_passage_new    = n_total - n_passage_cached

n_ner_cached = 0
if os.path.exists(ner_json):
    import json as _json
    with open(ner_json) as _f:
        _ner = _json.load(_f)
    n_ner_cached = len(_ner.get("passage_hash_id_to_entities", {}))

print(f"Index status:")
print(f"  Passages : {n_passage_cached} / {n_total} cached  ({n_passage_new} to embed)")
print(f"  NER      : {n_ner_cached} / {n_total} cached  ({max(0, n_total - n_ner_cached)} to process)")

if n_passage_new == 0 and n_ner_cached >= n_total:
    print("\nAll data is cached — index() will only rebuild the graph (no API calls, no progress bars expected).")
else:
    print(f"\nProgress bars will appear below for embedding and NER steps.")

# index() is always called — it is resumable internally via load_existing_data()
linearrag.index(indexed_texts)
print("\nIndexing complete")

# ── Rebuild hash_id → original_id mapping from the populated embedding store ──
hash_to_orig_id = {}
for stored_text, hash_id in linearrag.passage_embedding_store.text_to_hash_id.items():
    orig_id = text_to_orig_id.get(stored_text)
    if orig_id is not None:
        hash_to_orig_id[hash_id] = orig_id

n_mapped = len(hash_to_orig_id)
print(f"  Mapped {n_mapped} / {len(linearrag.passage_embedding_store.hash_ids)} passage hash IDs to original document IDs")
if n_mapped < len(linearrag.passage_embedding_store.hash_ids):
    print("  WARNING: some passages could not be mapped — check indexed_texts construction")


Index status:
  Passages : 5260 / 5260 cached  (0 to embed)
  NER      : 5260 / 5260 cached  (0 to process)

All data is cached — index() will only rebuild the graph (no API calls, no progress bars expected).
[passage] All 5260 texts already indexed.
[sentence] All 119623 texts already indexed.
[entity] All 95114 texts already indexed.

Indexing complete
  Mapped 5260 / 5260 passage hash IDs to original document IDs


In [7]:
# ============================================================
# Cell 7 — Load question datasets
# ============================================================
print("Loading question datasets...\n")

dfs = []
for ds in QUESTION_DATASETS:
    path = os.path.join(QUESTIONS_DIR, ds["file"])
    if not os.path.exists(path):
        print(f"  WARNING: {path} not found — skipping")
        continue
    df_ds = create_questions_dataset(
        jsonl_path=path,
        column_mapping=ds["column_mapping"],
        source1_name=ds["source1_name"],
        source2_name=ds["source2_name"],
        question_type=ds["question_type"],
    )
    df_ds["dataset_name"] = ds["name"]
    dfs.append(df_ds)
    print()

questions_df = pd.concat(dfs, ignore_index=True).reset_index(drop=True)

# Build unique_id (same format as perform_retrieval in retrieval_utils.py)
questions_df["unique_id"] = questions_df.apply(
    lambda row: (
        f"{row['source_1']}|{row['source_2']}|"
        f"{row['id_1']}|{row['id_2']}|"
        f"{row['question_index']}|{row.name}"
    ),
    axis=1,
)

print(f"Total questions : {len(questions_df)}")
print(f"Question types  : {questions_df['question_type'].value_counts().to_dict()}")

Loading question datasets...

Created dataset with 220 questions from 77 original entries

Created dataset with 199 questions from 79 original entries

Created dataset with 152 questions from 79 original entries

Created dataset with 192 questions from 47 original entries

Created dataset with 122 questions from 48 original entries

Total questions : 885
Question types  : {'MH - Generic': 541, 'MH - Comparative': 192, 'MH - BridgeEntity': 152}


In [8]:
# ============================================================
# Cell 8 — Retrieval
# Appends incrementally to OUTPUT_FILE; already-retrieved
# questions (matched by unique_id) are skipped automatically.
# ============================================================
existing_ids = get_existing_ids(OUTPUT_FILE)
print(f"Already retrieved : {len(existing_ids)} questions")

remaining_df = questions_df[~questions_df["unique_id"].isin(existing_ids)].copy()
print(f"To retrieve       : {len(remaining_df)} questions")

if len(remaining_df) == 0:
    print("Nothing to do — all questions already retrieved.")
else:
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "a", encoding="utf-8") as out_f:
        for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="LinearRAG retrieve"):
            question  = row["question"]
            gold_ids  = row["gold_ids"]

            # LinearRAG expects a list of {question, answer} dicts
            results = linearrag.retrieve([{"question": question, "answer": row["llm_answer"]}])
            result  = results[0]

            # Map internal hash IDs back to original document IDs
            passage_hash_ids = result.get("sorted_passage_hash_ids", [])
            retrieved_ids = [
                hash_to_orig_id.get(h, f"unknown:{h[:8]}") for h in passage_hash_ids
            ]

            gold_ranks  = find_gold_ranks(retrieved_ids, gold_ids, not_found_value=-1)
            dual_status = (
                "both_found"
                if all(0 <= rank < RETRIEVAL_TOP_K for rank in gold_ranks)
                else "incomplete"
            )

            record = {
                "unique_id":             row["unique_id"],
                "id_1":                  row["id_1"],
                "id_2":                  row["id_2"],
                "question":              question,
                "llm_answer":            row["llm_answer"],
                "question_type":         row["question_type"],
                "source_1":              row["source_1"],
                "source_2":              row["source_2"],
                "doc_1":                 row["doc_1"],
                "doc_2":                 row["doc_2"],
                "gold_ids":              gold_ids,
                "retrieved_ids":         retrieved_ids,
                "gold_ranks":            gold_ranks,
                "dual_retrieval_status": dual_status,
            }
            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"\nRetrieval complete. Results saved to {OUTPUT_FILE}")

Already retrieved : 0 questions
To retrieve       : 885 questions


LinearRAG retrieve: 100%|██████████| 885/885 [25:57<00:00,  1.76s/it]


Retrieval complete. Results saved to c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\retrieval\linearrag_retrieval_results.jsonl


In [9]:
# ============================================================
# Cell 9 — Load results & global evaluation
# ============================================================
print(f"Loading results from {OUTPUT_FILE}...")
records = []
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df_result = pd.DataFrame(records)
print(f"  {len(df_result)} results loaded")

# Global evaluation
metrics = evaluate_and_print_metrics(
    df_result,
    dataset_name="LinearRAG — 1887 Combined",
    gold_ranks_column="gold_ranks",
    k_values=K_VALUES,
)

Loading results from c:\Users\plathoon\Desktop\Aurelien\ORDER_EMNLP2026\data\retrieval\linearrag_retrieval_results.jsonl...
  885 results loaded

EVALUATION RESULTS FOR: LINEARRAG — 1887 COMBINED

Computing recall metrics...


100%|██████████| 885/885 [00:00<00:00, 6318.37it/s]



Recall Metrics for 885 questions:
  recall@1: 0.075
  recall@2: 0.132
  recall@3: 0.182
  recall@5: 0.270
  recall@10: 0.401


Computing Accuracy: 100%|██████████| 885/885 [00:00<00:00, 6610.28it/s]



Accuracy Metrics for 885 questions:
  accuracy@1: 0.000
  accuracy@2: 0.017
  accuracy@3: 0.041
  accuracy@5: 0.092
  accuracy@10: 0.200


Computing MRR: 100%|██████████| 885/885 [00:00<00:00, 3172.62it/s]


MRR Metrics for 885 questions:
  mrr@1: 0.075
  mrr@2: 0.103
  mrr@3: 0.120
  mrr@5: 0.140
  mrr@10: 0.157


In [10]:
# ============================================================
# Cell 10 — Breakdown by question type
# ============================================================
print("=" * 60)
print("BREAKDOWN BY QUESTION TYPE")
print("=" * 60)

for qtype in sorted(df_result["question_type"].unique()):
    subset = df_result[df_result["question_type"] == qtype]
    evaluate_and_print_metrics(
        subset,
        dataset_name=f"LinearRAG — {qtype} (n={len(subset)})",
        gold_ranks_column="gold_ranks",
        k_values=K_VALUES,
    )

BREAKDOWN BY QUESTION TYPE

EVALUATION RESULTS FOR: LINEARRAG — MH - BRIDGEENTITY (N=152)

Computing recall metrics...


100%|██████████| 152/152 [00:00<00:00, 22620.43it/s]



Recall Metrics for 152 questions:
  recall@1: 0.112
  recall@2: 0.220
  recall@3: 0.276
  recall@5: 0.388
  recall@10: 0.536


Computing Accuracy: 100%|██████████| 152/152 [00:00<00:00, 19005.34it/s]



Accuracy Metrics for 152 questions:
  accuracy@1: 0.000
  accuracy@2: 0.046
  accuracy@3: 0.066
  accuracy@5: 0.145
  accuracy@10: 0.289


...


Recall Metrics for 541 questions:
  recall@1: 0.081
  recall@2: 0.136
  recall@3: 0.196
  recall@5: 0.296
  recall@10: 0.440


Computing Accuracy: 100%|██████████| 541/541 [00:00<00:00, 13409.84it/s]



Accuracy Metrics for 541 questions:
  accuracy@1: 0.000
  accuracy@2: 0.013
  accuracy@3: 0.044
  accuracy@5: 0.104
  accuracy@10: 0.220


Computing MRR: 100%|██████████| 541/541 [00:00<00:00, 4896.43it/s]


MRR Metrics for 541 questions:
  mrr@1: 0.081
  mrr@2: 0.109
  mrr@3: 0.129
  mrr@5: 0.151
  mrr@10: 0.170


In [11]:
# ============================================================
# Cell 11 — Breakdown by source pair
# ============================================================
print("=" * 60)
print("BREAKDOWN BY SOURCE PAIR")
print("=" * 60)

df_result["source_pair"] = df_result["source_1"] + " × " + df_result["source_2"]
for pair in sorted(df_result["source_pair"].unique()):
    subset = df_result[df_result["source_pair"] == pair]
    evaluate_and_print_metrics(
        subset,
        dataset_name=f"LinearRAG — {pair} (n={len(subset)})",
        gold_ranks_column="gold_ranks",
        k_values=K_VALUES,
    )

BREAKDOWN BY SOURCE PAIR

EVALUATION RESULTS FOR: LINEARRAG — L'INTRANSIGEANT × LES DÉBATS (N=351)

Computing recall metrics...


100%|██████████| 351/351 [00:00<00:00, 18017.83it/s]



Recall Metrics for 351 questions:
  recall@1: 0.113
  recall@2: 0.188
  recall@3: 0.251
  recall@5: 0.365
  recall@10: 0.509


Computing Accuracy: 100%|██████████| 351/351 [00:00<00:00, 18474.56it/s]



Accuracy Metrics for 351 questions:
  accuracy@1: 0.000
  accuracy@2: 0.031
  accuracy@3: 0.066
  accuracy@5: 0.120
  accuracy@10: 0.262


...


Recall Metrics for 220 questions:
  recall@1: 0.086
  recall@2: 0.166
  recall@3: 0.243
  recall@5: 0.357
  recall@10: 0.509


Computing Accuracy: 100%|██████████| 220/220 [00:00<00:00, 11298.90it/s]



Accuracy Metrics for 220 questions:
  accuracy@1: 0.000
  accuracy@2: 0.014
  accuracy@3: 0.045
  accuracy@5: 0.145
  accuracy@10: 0.259


Computing MRR: 100%|██████████| 220/220 [00:00<00:00, 2790.75it/s]


MRR Metrics for 220 questions:
  mrr@1: 0.086
  mrr@2: 0.126
  mrr@3: 0.152
  mrr@5: 0.178
  mrr@10: 0.198


In [ ]:
# ============================================================
# Cell 12 — Summary table (Recall / Accuracy / MRR)
# ============================================================
subsets = [("All", df_result)] + [
    (qtype, df_result[df_result["question_type"] == qtype])
    for qtype in sorted(df_result["question_type"].unique())
]

summary_rows = []
for label, subset in subsets:
    if len(subset) == 0:
        continue
    r = compute_recall_metrics_dataframe(subset,   k_values=K_VALUES, gold_ranks_column="gold_ranks")
    a = compute_accuracy_at_k(subset,              k_values=K_VALUES, gold_ranks_column="gold_ranks")
    m = compute_mrr_at_k(subset,                   k_values=K_VALUES, gold_ranks_column="gold_ranks")
    row = {"Subset": label, "N": len(subset)}
    for k in K_VALUES:
        row[f"R@{k}"]   = f"{np.mean(r[f'recall@{k}']):.3f}"
        row[f"Acc@{k}"] = f"{np.mean(a[f'accuracy@{k}']):.3f}"
        row[f"MRR@{k}"] = f"{np.mean(m[f'mrr@{k}']):.3f}"
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).set_index("Subset")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
display(df_summary)